# Ladder — evaluate the trained adapter

Scores `Ladder-3B` against the untouched base model by **running generated
programs against real Codeforces test cases**. No model-as-judge.

**Runtime → Change runtime type → T4 GPU**, then Run All. It will ask for a
Hugging Face token (read access is enough).

Runs on **div2 A/B problems**. A first attempt scored the full difficulty range
and most responses came back with no code at all: a 3B reasons past the token
budget on hard problems and gets cut off before writing any. That measures the
budget, not the model. A/B problems need far less reasoning, so the budget is
sufficient and the score can actually move.

The slice is part of the result — a pass@1 on div2 A/B is not comparable to one
over all of Codeforces.

**Colab disconnects.** Results are written after every problem and a re-run
resumes, so if it drops just run the cells again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Not %%capture: hiding pip output means a failed install surfaces later as a
# confusing ImportError instead of the real error.
!pip install -U unsloth unsloth_zoo
!pip install -U bitsandbytes datasets

# Always a fresh clone. Reconnecting to an existing Colab runtime leaves the
# previous /content/ladder in place, and `git clone` into an existing
# directory fails -- quietly, if the output is suppressed -- so the session
# then runs against a stale checkout missing whatever was just added.
!rm -rf /content/ladder
!git clone https://github.com/NiLabs-Org/ladder.git /content/ladder
!ls /content/ladder/configs/


In [ ]:
import sys
sys.path.insert(0, "/content/ladder/src")

from huggingface_hub import login, snapshot_download

# Prefer a Colab secret named HF_TOKEN; otherwise just ask. getpass keeps the
# token out of the notebook source and out of its saved output.
token = None
try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
except Exception:
    from getpass import getpass

    token = getpass("HF token (read access): ").strip()

login(token)
ADAPTER = snapshot_download(repo_id="ndemoss28/Ladder-3B", repo_type="model")
print("adapter:", ADAPTER)

## Problems

Loads a wide pool from the held-out split, then keeps div2 A/B.

In [ ]:
import os

from ladder.config import load_config
from ladder.eval.tasks import load_problems

cfg = load_config("/content/ladder/configs/eval-easy.yaml")

OUT = "/content/drive/MyDrive/ladder" if os.path.isdir("/content/drive/MyDrive") else "/content/out"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT)

problems = load_problems(cfg.eval, cfg.data)
print(f"{len(problems)} problems")
for p in problems[:5]:
    print(" ", p.problem_id, p.title[:55])

Optional, so results survive a disconnect:

```python
from google.colab import drive; drive.mount('/content/drive')
```

## Score the base model

This is what the fine-tune has to beat, so it runs first.

In [ ]:
import gc

import torch

from ladder.eval.runner import evaluate
from ladder.infer import load_for_inference, make_generator


def score(adapter, label):
    cfg.eval.results_path = f"{OUT}/easy-{label}.json"
    model, tok = load_for_inference(cfg, adapter)
    try:
        return evaluate(make_generator(model, tok, cfg), cfg, problems)
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()


base = score(None, "base")
base["metrics"]

## Score the fine-tune

Same problems, same prompt, same decoding. Only the adapter differs.

In [ ]:
tuned = score(ADAPTER, "tuned")
tuned["metrics"]

## Results

In [ ]:
b = base["metrics"]["pass@1"]
t = tuned["metrics"]["pass@1"]

print(f"{'model':<34} {'pass@1':>8}")
print(f"{'Qwen2.5-Coder-3B-Instruct (base)':<34} {b:>8.3f}")
print(f"{'Ladder-3B':<34} {t:>8.3f}")
print(f"{'delta':<34} {t - b:>+8.3f}")
print()
print("problems      :", tuned["n_problems"], "(div2 A/B)")
print("base verdicts :", base["verdicts"])
print("tuned verdicts:", tuned["verdicts"])
print()
print("Send these back to record the run:")
print(" ", OUT + "/easy-base.json")
print(" ", OUT + "/easy-tuned.json")